# ECB SPF — first look at the HICP panel

Loads all ECB SPF individual-forecast CSVs from `data/ecb/individual_forecasts/`,
extracts the year-on-year HICP inflation expectations, classifies each
`TARGET_PERIOD` string, and tabulates the panel structure (forecaster IDs,
horizons, coverage) before any rank-CI computation. This mirrors the
`EDA.ipynb` step on the Philly side.

To compute MSE-style rank CIs we still need a realization series — a Eurostat
HICP first-release vintage indexed by `target_period`. Once that exists, the
downstream pipeline is identical to the Philly notebooks (`select_top_forecasters`
→ `rank_ci_stepwise_pairwise` → `tau_best_pairwise`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rankci import (
    rank_ci_stepwise_pairwise,
    rank_ci_marginal_pairwise,
    tau_best_pairwise,
    tau_best_from_rank_ci,
    compute_pairwise,
)
from rankci.data.ecb import (
    INDICATORS,
    load_ecb_csv,
    load_ecb_spf,
    list_survey_csvs,
    add_horizon,
    error_panel,
    load_hicp_realized,
    hicp_realized_by_target_period,
)
from rankci.data.panel import select_top_forecasters

DATA_DIR  = '../../data/ecb/individual_forecasts'
HICP_REAL = '../../data/ecb/prc_hicp_manr__custom_21709551_spreadsheet.xlsx'

## 1. Survey vintages on disk

In [ ]:
files = list_survey_csvs(DATA_DIR)
print(f'{len(files)} survey CSVs, {files[0].name} → {files[-1].name}')
print('Indicator sections expected in each CSV:')
for k, v in INDICATORS.items():
    print(f'  {k:9s} {v}')

## 2. Inspect one vintage

In [ ]:
one = load_ecb_csv(files[0])
print(f'{files[0].name} — {len(one)} rows')
print('Indicator counts:')
print(one.indicator.value_counts())
print('\nTarget-period kinds:')
print(one.target_kind.value_counts())
one.head()

## 3. Load HICP across all surveys

In [ ]:
hicp = load_ecb_spf(DATA_DIR, indicators=['HICP'])
hicp = add_horizon(hicp)
print(f'HICP rows: {len(hicp):,}')
print(f'Unique forecaster IDs: {hicp.forecaster_id.nunique()}')
print(f'Survey coverage: ({hicp.survey_year.min()},Q{hicp.survey_quarter.min()}) '
      f'→ ({hicp.survey_year.max()},Q{hicp.survey_quarter.max()})')
print(f'Target kinds: {dict(hicp.target_kind.value_counts())}')

## 4. Horizon distribution

Horizons are signed integer quarters from survey to target. The ECB short-run
panel has 0–7-quarter horizons; the long-term (5-year-ahead) block sits at
~18–21 quarters.

In [ ]:
h_dist = (hicp[hicp.horizon_q.notna()]
          .groupby('horizon_q').size()
          .rename('n_forecasts'))
print(h_dist.to_string())

## 5. Forecaster coverage

How many surveys does each forecaster show up in (any horizon)?

In [ ]:
surveys_per_fid = (hicp.drop_duplicates(['forecaster_id', 'survey_year', 'survey_quarter'])
                       .groupby('forecaster_id').size()
                       .sort_values(ascending=False))
print(f'Forecasters with ≥ 20 survey appearances: {(surveys_per_fid >= 20).sum()}')
print(f'Forecasters with ≥ 50 survey appearances: {(surveys_per_fid >= 50).sum()}')
print(f'Forecasters with ≥ 80 survey appearances: {(surveys_per_fid >= 80).sum()}')
print('\nTop 15 by coverage:')
print(surveys_per_fid.head(15))

## 6. Realized HICP from Eurostat

`prc_hicp_manr__custom_21709551_spreadsheet.xlsx` is the Eurostat dataset of
monthly year-on-year HICP inflation for the Euro area (1997-01 onwards).
`hicp_realized_by_target_period` converts it into a Series keyed by exactly
the ECB SPF `TARGET_PERIOD` strings, so it slots into `error_panel` directly:

- `"YYYY"` → mean of 12 monthly YoY rates that year
- `"YYYYMon"` → the YoY rate of that specific month
- `"YYYYQq"` → mean of the 3 monthly YoY rates in that quarter

In [ ]:
monthly = load_hicp_realized(HICP_REAL)
print(f'Eurostat HICP YoY: {len(monthly)} monthly obs, '
      f'{monthly.index.min()} → {monthly.index.max()}')

realized = hicp_realized_by_target_period(monthly)
print(f'Realized series, total target_period keys: {len(realized)}')

# A few sanity-check spot reads
for k in ['2003', '2003Sep', '2003Q4', '2008', '2020', '2020Mar', '2020Q2', '2025']:
    val = realized.get(k)
    print(f'  {k:10s} -> {val:.2f}' if val is not None else f'  {k:10s} -> (missing)')

## 7. Squared-error panel and rank CIs

We compute MSE rankings on the **annual** target ("YYYY") at horizon `h=3`
quarters (one-year-ahead forecasts of the next-calendar-year average inflation
rate, which is the canonical ECB SPF block). Then `select_top_forecasters` →
`rank_ci_stepwise_pairwise`, exactly as on the Philly notebooks.

In [ ]:
TARGET_KIND = 'year'    # 'year' | 'month' | 'quarter'
HORIZON_Q   = 3         # quarters from survey to target
METRIC      = 'squared' # 'squared' or 'absolute'
N           = 8
MIN_OBS     = 15

X_wide = error_panel(
    hicp, realized,
    indicator='HICP', target_kind=TARGET_KIND,
    horizon_q=HORIZON_Q, metric=METRIC,
)
print(f'Wide panel: {X_wide.shape[0]} targets × {X_wide.shape[1]} forecasters')

obs_counts = X_wide.notna().sum()
print(f'  ≥ {MIN_OBS} obs: {(obs_counts >= MIN_OBS).sum()} forecasters')
print(f'  ≥ 20 obs: {(obs_counts >= 20).sum()} forecasters')

X_panel = select_top_forecasters(X_wide, N=N, min_obs=MIN_OBS)
print(f'\nSelected top-{N}: {X_panel.shape[0]} targets, IDs={X_panel.columns.tolist()}')

In [ ]:
X = X_panel.values
population_ids = X_panel.columns.tolist()

out = rank_ci_stepwise_pairwise(X, alpha=0.05, B=5000, seed=42)

results = pd.DataFrame({
    'ID':       population_ids,
    'MSE':      out['theta_hat'].round(4),
    'RMSE':     np.sqrt(out['theta_hat']).round(4),
    'CI_lower': out['rank_ci'][:, 0],
    'CI_upper': out['rank_ci'][:, 1],
}).sort_values('MSE')
results.index = range(1, len(results) + 1)
results.index.name = 'Rank'
results

## 8. Worst-target diagnostic

Which years drive the squared error? Useful to confirm 2008 / 2020-2022
(GFC, COVID, post-COVID inflation surge) dominate the loss, the same
crisis-tail pattern that makes the Philly CIs uninformative.

In [ ]:
period_mse = X_panel.mean(axis=1)
print('Top 10 targets with highest average squared error:')
print(period_mse.nlargest(10).round(3))

plt.figure(figsize=(11, 3.5))
plt.plot(X_panel.index, period_mse, marker='o', linewidth=1, markersize=4)
plt.xticks(rotation=45)
plt.ylabel(f'Avg squared error ({METRIC})')
plt.title(f'Avg squared HICP error, top-{N} ECB forecasters, '
          f'{TARGET_KIND}/h={HORIZON_Q}q')
plt.tight_layout()
plt.show()

## 9. Marginal CIs

For comparison: per-forecaster CIs that control coverage only marginally
(no joint family-wise error control). They are tighter than the simultaneous
stepwise CIs but don't deliver a joint guarantee.

In [ ]:
out_marg = rank_ci_marginal_pairwise(X, alpha=0.1, B=5000, seed=42)

results_marg = pd.DataFrame({
    'ID':       population_ids,
    'MSE':      out_marg['theta_hat'].round(4),
    'RMSE':     np.sqrt(out_marg['theta_hat']).round(4),
    'cv_j':     out_marg['critical_values'].round(3),
    'CI_lower': out_marg['rank_ci'][:, 0],
    'CI_upper': out_marg['rank_ci'][:, 1],
}).sort_values('MSE')
results_marg.index = range(1, len(results_marg) + 1)
results_marg.index.name = 'Rank'
results_marg

## 10. τ-best confidence set

Same Algorithm 3.3 as in `Reports/tau_best_report.tex`, applied to ECB HICP.
For each $\tau \in \{1,2,3\}$ we report the forecasters that *cannot* be ruled
out of the truly top-$\tau$ at $\alpha = 0.2$.

In [ ]:
ALPHA_TAU = 0.2
B_TAU     = 5000
SEED_TAU  = 42

out_rank = rank_ci_stepwise_pairwise(
    X, alpha=ALPHA_TAU, B=B_TAU, seed=SEED_TAU, verbose=False,
)

rows = []
for tau in [1, 2, 3]:
    res   = tau_best_pairwise(X, tau=tau, alpha=ALPHA_TAU, B=B_TAU,
                              seed=SEED_TAU, verbose=False)
    naive = tau_best_from_rank_ci(out_rank['rank_ci'], tau=tau)
    rows.append({
        'tau':           tau,
        'n_direct':      int(res['n_in_set']),
        'set_direct':    [population_ids[j] for j, v in enumerate(res['tau_best_set']) if v],
        'n_naive':       int(naive.sum()),
        'set_naive':     [population_ids[j] for j, v in enumerate(naive) if v],
        'max_test_stat': round(float(np.nanmax(res['test_stats'])), 3),
    })

tau_sweep = pd.DataFrame(rows).set_index('tau')
print(f'alpha = {ALPHA_TAU}, B = {B_TAU}, p = {len(population_ids)}')
print(f'Forecaster IDs: {population_ids}')
print()
tau_sweep